In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

In [ ]:
!nvidia-smi

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    if files:
        print(root)
        print(files[:5])

In [ ]:
import pandas as pd

BASE = "/kaggle/input/datasets/gillesokhin/nutrition5k-dataset"

nutrition = pd.read_csv(f"{BASE}/dish_nutrition_values.csv")

print(nutrition.columns.tolist())
display(nutrition.head())

In [ ]:
import os

nutrition["image_path"] = nutrition["dish_id"].apply(
    lambda x: f"{BASE}/imagery/realsense_overhead/{x}/rgb.png"
)

# chỉ giữ món thực sự có ảnh RGB
nutrition = nutrition[
    nutrition["image_path"].apply(os.path.exists)
].reset_index(drop=True)

print("Số món có ảnh:", len(nutrition))

display(
    nutrition[
        ["dish_id", "image_path", "total_calories"]
    ].head()
)

In [ ]:
print(nutrition.columns.tolist())
display(nutrition.head())

In [ ]:
df = nutrition[
    ["dish_id", "image_path", "calories"]
].copy()

display(df.head())
print("Tổng số mẫu:", len(df))

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42
)

print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

class NutritionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")
        calories = torch.tensor(row["calories"], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, calories

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
train_dataset = NutritionDataset(train_df, train_transform)
val_dataset = NutritionDataset(val_df, val_transform)
test_dataset = NutritionDataset(test_df, val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(len(train_dataset), len(val_dataset), len(test_dataset))

In [ ]:
images, calories = next(iter(train_loader))

print("Images:", images.shape)
print("Calories:", calories.shape)
print("Ví dụ calories:", calories[:5])

In [ ]:
# Lưu split cố định để mọi experiment sau dùng cùng dữ liệu
train_df.to_csv("/kaggle/working/train.csv", index=False)
val_df.to_csv("/kaggle/working/val.csv", index=False)
test_df.to_csv("/kaggle/working/test.csv", index=False)

print("Saved fixed splits:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


In [ ]:
# Thống kê mô tả target calories
def calorie_stats(dataframe, split_name):
    return {
        "split": split_name,
        "n_samples": len(dataframe),
        "min_kcal": dataframe["calories"].min(),
        "max_kcal": dataframe["calories"].max(),
        "mean_kcal": dataframe["calories"].mean(),
        "std_kcal": dataframe["calories"].std(),
        "median_kcal": dataframe["calories"].median(),
    }

stats_df = pd.DataFrame([
    calorie_stats(train_df, "Train"),
    calorie_stats(val_df, "Validation"),
    calorie_stats(test_df, "Test"),
    calorie_stats(pd.concat([train_df, val_df, test_df]), "Total"),
])

display(stats_df.round(2))
stats_df.to_csv("/kaggle/working/dataset_calorie_statistics.csv", index=False)


# EXPERIMENTS

Phần chuẩn bị dữ liệu phía trên được giữ cố định.  
Thêm các experiment mới từ đây trở xuống để bảo đảm tất cả thí nghiệm dùng cùng dataset split.
